# Paper 2 · Langkah 4: peta atribusi dan uji kesetiaan (RQ2)

Notebook ini membuka "mata" detektor:
1. Membuat peta **Grad-CAM** dan **occlusion** untuk setiap gambar, memakai model dari fold tempat gambar itu diuji.
2. Menguji apakah peta itu **setia** pada model: uji penghapusan (deletion), uji pengacakan model, dan cek bias tengah.
3. Melatih ulang dua model lintas generator dan membuat peta untuk gambar dari generator yang tidak dikenal model (bahan RQ3).

Dasar peta: kondisi **riwayat_disamakan**. Kondisi **normal** dibuat petanya saja sebagai pembanding.

**Sebelum Run all:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Hasil tiap bagian langsung disimpan ke Drive. Kalau sesi terputus, klik Run all lagi; bagian yang sudah selesai dilewati.

## 1. Persiapan dan cek GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, io, json, time, copy, shutil, zipfile, random, platform
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision.transforms.functional import gaussian_blur
from PIL import Image
import PIL

# Pengaturan (jangan diubah tanpa mencatatnya di log kerja)
SEED = 42
INPUT = 224
PATCH, STRIDE = 32, 16          # occlusion utama
PATCH2, STRIDE2 = 48, 24        # occlusion pembanding (uji ukuran kotak)
TOP = 0.20                      # area sorotan R = 20% piksel teratas
N_RANDOM = 10                   # jumlah peta acak per gambar
BLUR_KERNEL, BLUR_SIGMA = 21, 7.0
SUBSET = 0.20                   # porsi gambar untuk uji yang mahal
# Latihan (sama dengan langkah 3)
MAX_EPOCHS, PATIENCE, BATCH, LR, LR_MIN, PRETRAINED = 30, 5, 32, 1e-4, 1e-6, True

PROJECT = Path('/content/drive/MyDrive/Paper2')
RAW = PROJECT / 'data_asli' / 'ai_artbench_sample'
NOTES = PROJECT / 'catatan'
RESULTS = PROJECT / 'hasil'
RUNS = RESULTS / 'detektor_runs'
MODELS = RESULTS / 'models'
MAPS = RESULTS / 'peta'
CACHE = PROJECT / 'data_olahan'
LOCAL = Path('/content/work')
for p in [RUNS, MODELS, MAPS, CACHE, LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'TIDAK ADA'
print('GPU:', gpu)
assert device == 'cuda', 'GPU belum aktif. Buka Runtime → Change runtime type → pilih T4 GPU → Save, lalu Run all lagi.'
USE_AMP = device == 'cuda'

env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\ntorch: {torch.__version__}\n'
       f'torchvision: {torchvision.__version__}\npillow: {PIL.__version__}\ngpu: {gpu}\n'
       f'occlusion: kotak {PATCH}, langkah {STRIDE}; pembanding kotak {PATCH2}, langkah {STRIDE2}\n'
       f'area sorotan: {TOP}; peta acak: {N_RANDOM}; blur: kernel {BLUR_KERNEL}, sigma {BLUR_SIGMA}; subset: {SUBSET}\n')
(NOTES / 'lingkungan_langkah4.txt').write_text(env)
print(env)

## 2. Data: dua kondisi dimuat ke memori

In [ ]:
manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv')
assert len(manifest) == 3000 and 'fold' in manifest.columns, 'Jalankan notebook langkah 2 dulu.'
manifest['y'] = (manifest['label'] == 'ai').astype(int)

local_raw = LOCAL / 'raw'
raw_zip = CACHE / 'ai_artbench_sample_raw.zip'
t0 = time.time()
if not local_raw.exists():
    assert raw_zip.exists(), 'File zip data belum ada. Jalankan notebook langkah 3 dulu.'
    shutil.copy(raw_zip, LOCAL / 'raw.zip')
    with zipfile.ZipFile(LOCAL / 'raw.zip') as z:
        z.extractall(local_raw)
manifest['raw_path'] = manifest.apply(
    lambda r: str(local_raw / r['source'] / r['style'] / Path(r['file_drive']).name), axis=1)
missing = [p for p in manifest['raw_path'] if not os.path.exists(p)]
assert not missing, f'{len(missing)} file tidak ditemukan, contoh: {missing[:3]}'

def to_norm(im):
    im = im.convert('RGB')
    w, h = im.size
    if w != h:
        s = min(w, h); left, top = (w - s) // 2, (h - s) // 2
        im = im.crop((left, top, left + s, top + s))
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    buf = io.BytesIO()
    im.save(buf, 'JPEG', quality=90, subsampling=0, optimize=False)
    return buf.getvalue()

def make_condition(row, cond):
    with Image.open(row.raw_path) as im:
        im.load()
        if cond == 'riwayat_disamakan' and row.source == 'human':
            buf = io.BytesIO()
            im.convert('RGB').save(buf, 'JPEG', quality=75, subsampling=2, optimize=False)
            im = Image.open(io.BytesIO(buf.getvalue())); im.load()
        return Image.open(io.BytesIO(to_norm(im))).convert('RGB')

def load_condition(cond):
    def _one(row):
        im = make_condition(row, cond)
        if im.size != (INPUT, INPUT):
            im = im.resize((INPUT, INPUT), Image.BICUBIC)
        return np.asarray(im, dtype=np.uint8)
    with ThreadPoolExecutor(8) as ex:
        arr = list(ex.map(_one, manifest.itertuples()))
    return torch.from_numpy(np.stack(arr)).permute(0, 3, 1, 2).contiguous()

DATA = {c: load_condition(c) for c in ['riwayat_disamakan', 'normal']}
folds, src, y_all = manifest['fold'].values, manifest['source'].values, manifest['y'].values
strata_all = (manifest['source'] + '|' + manifest['style']).values
print(f'Data siap ({time.time() - t0:.0f} detik):', {k: tuple(v.shape) for k, v in DATA.items()})

## 3. Fungsi peta dan uji

- **Grad-CAM**: dihitung di blok konvolusi terakhir (layer4), untuk kelas AI dan kelas manusia.
- **Occlusion**: kotak 32 × 32 digeser dengan langkah 16 dan diisi warna rata-rata gambar; dicatat seberapa turun keyakinan model.
- **Deletion**: 5% sampai 50% piksel teratas menurut peta diganti (versi blur dan versi warna rata-rata), lalu dicatat keyakinan model. Dibandingkan dengan peta tengah dan peta acak.
- **Pengacakan model**: bobot lapisan akhir, lalu layer4, diacak; peta yang setia seharusnya berubah.

In [ ]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
NPIX = INPUT * INPUT
FRACS = torch.arange(1, 11, device=device) * 0.05

def norm_t(xb):
    return ((xb.to(device).float().div_(255) - MEAN) / STD).contiguous(memory_format=torch.channels_last)

def build_model(pretrained=False):
    w = torchvision.models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    m = torchvision.models.resnet50(weights=w)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(device).to(memory_format=torch.channels_last)

def load_model(name):
    m = build_model(False)
    m.load_state_dict(torch.load(MODELS / f'{name}.pt', map_location=device))
    return m.eval()

@torch.no_grad()
def probs(model, xn, bs=256):
    out = []
    for i in range(0, len(xn), bs):
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(xn[i:i + bs])
        out.append(torch.softmax(logits.float(), 1))
    return torch.cat(out)

def gradcam(model, xn):
    """Grad-CAM mentah di layer4 untuk kedua kelas: (B,2,h,w) dan probabilitas (B,2)."""
    store = {}
    hook = model.layer4.register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
    with torch.enable_grad():
        logits = model(xn)
        a = store['a']; cams = []
        for c in (0, 1):
            g = torch.autograd.grad(logits[:, c].sum(), a, retain_graph=(c == 0))[0]
            cams.append(torch.relu((g.mean((2, 3), keepdim=True) * a).sum(1)))
    hook.remove()
    return torch.stack(cams, 1).detach().float(), torch.softmax(logits.detach().float(), 1)

def make_masks(patch, stride):
    pos = list(range(0, INPUT - patch + 1, stride))
    M = torch.zeros(len(pos) ** 2, 1, INPUT, INPUT, dtype=torch.bool, device=device)
    k = 0
    for i in pos:
        for j in pos:
            M[k, 0, i:i + patch, j:j + patch] = True; k += 1
    return M, len(pos)
M1, N1 = make_masks(PATCH, STRIDE)
M2, N2 = make_masks(PATCH2, STRIDE2)

@torch.no_grad()
def occlusion(model, xn, M, n):
    """Penurunan probabilitas tiap kelas saat tiap kotak ditutup: (B,2,n,n)."""
    out = []
    for i in range(len(xn)):
        x = xn[i:i + 1]
        mu = x.mean((2, 3), keepdim=True)
        p0 = probs(model, x)[0]
        po = probs(model, torch.where(M, mu, x))
        out.append((p0[None] - po).T.reshape(2, n, n))
    return torch.stack(out)

def patch_to_cells(d, patch, stride):
    """Rata-ratakan kotak yang tumpang tindih menjadi peta sel."""
    k, n, C = patch // stride, d.shape[-1], INPUT // stride
    acc = torch.zeros(*d.shape[:-2], C, C, device=d.device)
    cnt = torch.zeros(C, C, device=d.device)
    for a in range(k):
        for b in range(k):
            acc[..., a:a + n, b:b + n] += d; cnt[a:a + n, b:b + n] += 1
    return acc / cnt.clamp(min=1)

def up(m):
    return F.interpolate(m[:, None].float(), size=(INPUT, INPUT), mode='bilinear', align_corners=False)[:, 0]

def scale01(m):
    mn, mx = m.amin((1, 2), keepdim=True), m.amax((1, 2), keepdim=True)
    return (m - mn) / (mx - mn).clamp(min=1e-12)

def occ_map(d, patch, stride):
    cells = patch_to_cells(d, patch, stride).clamp(min=0)
    m = up(cells)
    return m / m.amax((1, 2), keepdim=True).clamp(min=1e-12), cells

def rank_desc(maps, gen):
    flat = maps.reshape(len(maps), -1)
    flat = flat + 1e-7 * torch.rand(flat.shape, device=device, generator=gen)   # pemecah nilai kembar
    order = flat.argsort(1, descending=True)
    rank = torch.empty_like(order)
    rank.scatter_(1, order, torch.arange(order.shape[1], device=device).expand_as(order))
    return rank

def spearman_t(a, b, gen):
    ra, rb = rank_desc(a, gen).float(), rank_desc(b, gen).float()
    ra, rb = ra - ra.mean(1, keepdim=True), rb - rb.mean(1, keepdim=True)
    den = (ra.norm(dim=1) * rb.norm(dim=1)).clamp(min=1e-12)
    rho = (ra * rb).sum(1) / den
    const = (a.reshape(len(a), -1).std(1) < 1e-12) | (b.reshape(len(b), -1).std(1) < 1e-12)
    rho[const] = float('nan')        # peta tanpa variasi: korelasi tidak terdefinisi
    return rho

yy, xx = torch.meshgrid(torch.arange(INPUT, device=device), torch.arange(INPUT, device=device), indexing='ij')
c0 = (INPUT - 1) / 2
CENTRE = 1 - torch.sqrt((yy - c0) ** 2 + (xx - c0) ** 2) / (c0 * 2 ** 0.5)       # peta tengah
side = int(round((TOP ** 0.5) * INPUT)); o = (INPUT - side) // 2
CENTRE_SQ = torch.zeros(INPUT, INPUT, dtype=torch.bool, device=device)
CENTRE_SQ[o:o + side, o:o + side] = True                                          # persegi tengah seluas R

@torch.no_grad()
def deletion(model, x, maps, c, p0, gen):
    """Luas di bawah kurva penghapusan (makin kecil makin setia) untuk tiap peta, versi blur dan warna rata-rata."""
    rank = rank_desc(maps, gen)
    thr = (FRACS * NPIX).long()
    mask = (rank[:, None, :] < thr[None, :, None]).reshape(-1, 1, INPUT, INPUT)
    out = {}
    for name, rep in (('blur', gaussian_blur(x, [BLUR_KERNEL, BLUR_KERNEL], [BLUR_SIGMA, BLUR_SIGMA])),
                      ('mean', x.mean((2, 3), keepdim=True))):
        p = probs(model, torch.where(mask, rep, x))[:, c].reshape(len(maps), -1)
        out[name] = (0.5 * p0 + p[:, :-1].sum(1) + 0.5 * p[:, -1]) / len(FRACS)
    return out

def randomised(model, level):
    m = copy.deepcopy(model)
    torch.manual_seed(SEED)
    m.fc.reset_parameters()
    if level == 'layer4':
        for mod in m.layer4.modules():
            if hasattr(mod, 'reset_parameters'):
                mod.reset_parameters()
    return m.eval()

def pick_subset(idx, seed):
    rng = np.random.default_rng(seed); keep = []
    for s in np.unique(strata_all[idx]):
        g = idx[strata_all[idx] == s]
        keep.extend(rng.choice(g, size=max(1, int(round(SUBSET * len(g)))), replace=False))
    return np.array(sorted(keep))
print('Fungsi siap. Posisi kotak occlusion:', N1, 'x', N1, '| pembanding:', N2, 'x', N2)

## 4. Latih ulang dua model lintas generator

Di langkah 3 bobot model lintas generator tidak disimpan, padahal sekarang dibutuhkan untuk membuat peta. Dua model dilatih ulang pada kondisi **riwayat_disamakan** dengan pengaturan yang sama, lalu bobot dan prediksinya disimpan. Angka lintas generator yang dipakai di paper diambil dari model yang disimpan ini.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def prep_train(xb):
    xb = xb.to(device, non_blocking=True).float().div_(255)
    flip = torch.rand(xb.size(0), device=device) < 0.5
    xb[flip] = xb[flip].flip(-1)
    return ((xb - MEAN) / STD).contiguous(memory_format=torch.channels_last)

@torch.no_grad()
def val_loss(model, X, idx):
    model.eval(); tot = 0.0
    for i in range(0, len(idx), 128):
        b = idx[i:i + 128]
        with torch.autocast(device_type=device, enabled=USE_AMP):
            logits = model(norm_t(X[b]))
        tot += F.cross_entropy(logits.float(), torch.as_tensor(y_all[b], device=device), reduction='sum').item()
    return tot / len(idx)

def train_model(X, tr_idx, va_idx, seed):
    set_seed(seed)
    model = build_model(PRETRAINED)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS, eta_min=LR_MIN)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    rng = np.random.default_rng(seed)
    best, best_state, bad, hist = float('inf'), None, 0, []
    for ep in range(MAX_EPOCHS):
        model.train(); perm = rng.permutation(tr_idx)
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=USE_AMP):
                loss = F.cross_entropy(model(prep_train(X[b])), torch.as_tensor(y_all[b], device=device))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        v = val_loss(model, X, va_idx); hist.append(round(v, 4))
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model.eval(), hist

def inner_split(tr_idx, seed):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.10, random_state=seed)
    try:
        a, b = next(sss.split(tr_idx, strata_all[tr_idx]))
    except ValueError:
        a, b = next(sss.split(tr_idx, src[tr_idx]))
    return tr_idx[a], tr_idx[b]

human_tr = np.where((src == 'human') & (folds != 4))[0]
human_te = np.where((src == 'human') & (folds == 4))[0]
SHORT = {'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
CROSS = {}
for train_gen, test_gen in [('latent_diffusion', 'stable_diffusion'), ('stable_diffusion', 'latent_diffusion')]:
    name = f'lintas2_riwayat_disamakan_latih{SHORT[train_gen]}_uji{SHORT[test_gen]}'
    te_idx = np.concatenate([human_te, np.where(src == test_gen)[0]])
    CROSS[name] = te_idx
    if (MODELS / f'{name}.pt').exists() and (RUNS / f'{name}.json').exists():
        print(f'{name}: sudah ada, dilewati.'); continue
    t0 = time.time()
    tr_idx = np.concatenate([human_tr, np.where(src == train_gen)[0]])
    tr, va = inner_split(tr_idx, SEED + 10)
    model, hist = train_model(DATA['riwayat_disamakan'], tr, va, SEED + 10)
    p = torch.cat([probs(model, norm_t(DATA['riwayat_disamakan'][te_idx[i:i + 128]])) for i in range(0, len(te_idx), 128)])[:, 1].cpu().numpy()
    pred = (p >= 0.5).astype(int); yt = y_all[te_idx]
    rec = {'nama': name, 'epoch': len(hist), 'val_loss': hist, 'detik': round(time.time() - t0),
           'akurasi': float(accuracy_score(yt, pred)), 'roc_auc': float(roc_auc_score(yt, p)),
           'benar_manusia': float((pred[yt == 0] == 0).mean()), 'ai_dikenali': float((pred[yt == 1] == 1).mean()),
           'prediksi': [{'sample_id': manifest['sample_id'].iat[i], 'p_ai': round(float(pi), 6)} for i, pi in zip(te_idx, p)]}
    torch.save(model.state_dict(), MODELS / f'{name}.pt')
    (RUNS / f'{name}.json').write_text(json.dumps(rec))
    print(f"{name}: akurasi {rec['akurasi']:.3f}, AUC {rec['roc_auc']:.3f}, AI dikenali {rec['ai_dikenali']:.3f}, {rec['epoch']} epoch, {rec['detik']} detik")
    del model; torch.cuda.empty_cache()

## 5. Buat peta dan jalankan uji

Tiga kelompok pekerjaan:
1. **riwayat_disamakan, 5 fold**: peta + deletion + pengacakan model + uji ukuran kotak.
2. **lintas generator, 2 model**: peta + deletion untuk gambar uji (karya manusia fold 4 dan gambar dari generator yang tidak dilatih).
3. **normal, 5 fold**: peta saja, sebagai pembanding.

Ini bagian terlama. Satu baris muncul tiap satu bagian selesai (12 bagian).

In [ ]:
def process(name, model_name, cond, idx, do_deletion, do_random):
    f = MAPS / f'{name}.npz'
    if f.exists():
        print(f'{name}: sudah selesai, dilewati.'); return
    t0 = time.time()
    model = load_model(model_name)
    X = DATA[cond]; n = len(idx)
    gen = torch.Generator(device=device); gen.manual_seed(SEED + sum(map(ord, name)))
    out = {k: [] for k in ['gc', 'occ', 'p', 'rho_gc_occ', 'iou_R', 'ciou_gc', 'ciou_occ']}
    dele = {f'{r}_{m}': [] for r in ('blur', 'mean') for m in ('gc', 'occ', 'centre', 'random')}
    C = INPUT // STRIDE; fm = INPUT // 32
    for s in range(0, n, 32):
        b = idx[s:s + 32]; xn = norm_t(X[b])
        cam, pr = gradcam(model, xn)
        occ = occlusion(model, xn, M1, N1)
        pred = pr.argmax(1); ar = torch.arange(len(b), device=device)
        gc224 = scale01(up(cam[ar, pred]))
        oc224, oc_cells = occ_map(occ[ar, pred], PATCH, STRIDE)
        gc_cells = F.adaptive_avg_pool2d(gc224[:, None], C)[:, 0]
        out['rho_gc_occ'].append(spearman_t(gc_cells, oc_cells, gen).cpu())
        Rg = (rank_desc(gc224, gen) < TOP * NPIX); Ro = (rank_desc(oc224, gen) < TOP * NPIX)
        out['iou_R'].append(((Rg & Ro).sum(1) / (Rg | Ro).sum(1).clamp(min=1)).cpu())
        csq = CENTRE_SQ.reshape(1, -1)
        out['ciou_gc'].append(((Rg & csq).sum(1) / (Rg | csq).sum(1)).cpu())
        out['ciou_occ'].append(((Ro & csq).sum(1) / (Ro | csq).sum(1)).cpu())
        out['gc'].append(cam.cpu()); out['occ'].append(occ.cpu()); out['p'].append(pr.cpu())
        if do_deletion:
            for i in range(len(b)):
                rnd = up(torch.rand(N_RANDOM, fm, fm, device=device, generator=gen))
                maps = torch.cat([gc224[i:i + 1], oc224[i:i + 1], CENTRE[None], rnd])
                res = deletion(model, xn[i:i + 1], maps, int(pred[i]), pr[i, pred[i]], gen)
                for r in ('blur', 'mean'):
                    a = res[r].cpu()
                    dele[f'{r}_gc'].append(a[0].item()); dele[f'{r}_occ'].append(a[1].item())
                    dele[f'{r}_centre'].append(a[2].item()); dele[f'{r}_random'].append(a[3:].mean().item())
    save = {k: torch.cat(v).numpy().astype(np.float32) for k, v in out.items()}
    save['sample_id'] = manifest['sample_id'].values[idx]
    save['pred'] = save['p'].argmax(1)
    for k, v in dele.items():
        if v: save['del_' + k] = np.array(v, dtype=np.float32)

    if do_random:
        pred_t = torch.as_tensor(save['pred'], device=device)
        sub = pick_subset(idx, SEED); pos = {v: i for i, v in enumerate(idx)}
        sub_loc = np.array([pos[v] for v in sub])
        save['subset_local'] = sub_loc
        gc_orig = torch.as_tensor(save['gc'])
        occ_orig = torch.as_tensor(save['occ'])
        for level in ('fc', 'layer4'):
            mr = randomised(model, level); rho = []
            for s in range(0, n, 32):
                b = idx[s:s + 32]; cam, _ = gradcam(mr, norm_t(X[b]))
                pt = pred_t[s:s + 32]; ar = torch.arange(len(b), device=device)
                rho.append(spearman_t(gc_orig[s:s + 32].to(device)[ar, pt], cam[ar, pt], gen).cpu())
            save[f'rand_{level}_gc'] = torch.cat(rho).numpy()
            rho = []
            for s in range(0, len(sub), 16):
                b = sub[s:s + 16]; loc = sub_loc[s:s + 16]
                oc = occlusion(mr, norm_t(X[b]), M1, N1)
                pt = pred_t[loc]; ar = torch.arange(len(b), device=device)
                rho.append(spearman_t(occ_orig[loc].to(device)[ar, pt], oc[ar, pt], gen).cpu())
            save[f'rand_{level}_occ'] = torch.cat(rho).numpy()
            del mr
        # Uji ukuran kotak occlusion pada subset
        rho = []
        for s in range(0, len(sub), 16):
            b = sub[s:s + 16]; loc = sub_loc[s:s + 16]
            oc2 = occlusion(model, norm_t(X[b]), M2, N2)
            pt = pred_t[loc]; ar = torch.arange(len(b), device=device)
            m2, _ = occ_map(oc2[ar, pt], PATCH2, STRIDE2)
            m1, _ = occ_map(occ_orig[loc].to(device)[ar, pt], PATCH, STRIDE)
            rho.append(spearman_t(F.adaptive_avg_pool2d(m1[:, None], C)[:, 0], F.adaptive_avg_pool2d(m2[:, None], C)[:, 0], gen).cpu())
        save['occ_patch_rho'] = torch.cat(rho).numpy()
    np.savez_compressed(f, **save)
    print(f'{name}: {n} gambar, {time.time() - t0:.0f} detik')
    del model; torch.cuda.empty_cache()

for k in range(5):
    process(f'riwayat_disamakan_fold{k}', f'riwayat_disamakan_fold{k}', 'riwayat_disamakan', np.where(folds == k)[0], True, True)
for name, te_idx in CROSS.items():
    process(name, name, 'riwayat_disamakan', te_idx, True, False)
for k in range(5):
    process(f'normal_fold{k}', f'normal_fold{k}', 'normal', np.where(folds == k)[0], False, False)

## 6. Ringkasan dan simpan hasil

In [ ]:
idx_of = {s: i for i, s in enumerate(manifest['sample_id'])}
rows = []
for f in sorted(MAPS.glob('*.npz')):
    z = np.load(f, allow_pickle=True); name = f.stem; n = len(z['sample_id'])
    d = pd.DataFrame({'bagian': name, 'sample_id': z['sample_id'], 'pred': z['pred'], 'p_ai': z['p'][:, 1],
                      'rho_gc_occ': z['rho_gc_occ'], 'iou_R': z['iou_R'], 'ciou_gc': z['ciou_gc'], 'ciou_occ': z['ciou_occ']})
    for k in z.files:
        if k.startswith('del_') or (k.startswith('rand_') and k.endswith('_gc')):
            d[k] = z[k]
    if 'subset_local' in z.files:
        for k in ('rand_fc_occ', 'rand_layer4_occ', 'occ_patch_rho'):
            col = np.full(n, np.nan); col[z['subset_local']] = z[k]; d[k] = col
    rows.append(d)
res = pd.concat(rows, ignore_index=True)
meta = manifest.set_index('sample_id')[['label', 'source', 'style', 'fold', 'y']]
res = res.join(meta, on='sample_id')
res['hasil'] = np.select([(res.y == 1) & (res.pred == 1), (res.y == 0) & (res.pred == 0), (res.y == 0) & (res.pred == 1)],
                         ['AI ditebak AI', 'manusia ditebak manusia', 'manusia ditebak AI'], 'AI ditebak manusia')
res['kelompok'] = np.where(res.bagian.str.startswith('lintas2'), 'lintas generator',
                   np.where(res.bagian.str.startswith('normal'), 'normal 5-fold', 'riwayat_disamakan 5-fold'))
res.to_csv(RESULTS / 'rq2_peta_per_gambar.csv', index=False)
print('Bagian selesai:', res.bagian.nunique(), 'dari 12')

def faithful(g):
    o = {'n': len(g)}
    for m in ('gc', 'occ'):
        ok = True
        for r in ('blur', 'mean'):
            a, c, rd = g[f'del_{r}_{m}'].mean(), g[f'del_{r}_centre'].mean(), g[f'del_{r}_random'].mean()
            o[f'{r}_{m}'] = a; ok &= (a < c) and (a < rd)
        o[f'lulus_{m}'] = 'ya' if ok else 'tidak'
    for r in ('blur', 'mean'):
        o[f'{r}_tengah'] = g[f'del_{r}_centre'].mean(); o[f'{r}_acak'] = g[f'del_{r}_random'].mean()
    return pd.Series(o)

cols = ['n', 'blur_gc', 'blur_occ', 'blur_tengah', 'blur_acak', 'mean_gc', 'mean_occ', 'mean_tengah', 'mean_acak', 'lulus_gc', 'lulus_occ']
d = res[res.kelompok != 'normal 5-fold']
t1 = d.groupby(['kelompok', 'source', 'hasil']).apply(faithful)[cols].round(3)
t1.to_csv(RESULTS / 'rq2_kesetiaan_deletion.csv')
print('\nUJI DELETION (angka = luas kurva; makin kecil makin setia; gc = Grad-CAM, occ = occlusion)')
display(t1)

h = res[res.kelompok == 'riwayat_disamakan 5-fold']
t2 = h.groupby('source')[['rand_fc_gc', 'rand_layer4_gc', 'rand_fc_occ', 'rand_layer4_occ']].mean().round(3)
print('\nUJI PENGACAKAN MODEL (korelasi dengan peta asli; makin dekat 0 makin baik)')
display(t2)

t3 = res.groupby(['kelompok', 'source'])[['rho_gc_occ', 'iou_R', 'ciou_gc', 'ciou_occ', 'occ_patch_rho']].mean().round(3)
t3.to_csv(RESULTS / 'rq2_kesepakatan_peta.csv')
print('\nKESEPAKATAN DUA PETA, BIAS TENGAH, DAN UJI UKURAN KOTAK')
print('rho_gc_occ = korelasi Grad-CAM dan occlusion; iou_R = tumpang tindih area sorotan;')
print('ciou = tumpang tindih dengan persegi tengah; occ_patch_rho = korelasi occlusion kotak 32 dan 48')
display(t3)

cr = []
for f in sorted(RUNS.glob('lintas2_*.json')):
    r = json.loads(f.read_text()); cr.append({k: r[k] for k in ('nama', 'akurasi', 'roc_auc', 'benar_manusia', 'ai_dikenali', 'epoch')})
print('\nMODEL LINTAS GENERATOR (yang disimpan)')
display(pd.DataFrame(cr).round(3))
print('\nJumlah gambar per kelompok dan hasil:')
display(res.groupby(['kelompok', 'source', 'hasil']).size().unstack(fill_value=0))
print('\nTersimpan di', RESULTS)

## Selesai

Kalau tertulis "Bagian selesai: 12 dari 12", semua beres. Tekan **Cmd+S**, lalu kabari Claude.

Kalau kurang dari 12, klik **Run all** lagi nanti. Bagian yang sudah selesai akan dilewati.